### Middleware

MIddleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following



- tracking agent behaviour with logging, analytics, and debugging
- transforming prompts, tool selection, and output formatting.
- adding retires, falbacks, and early termination logic.
- applying rate limits, guardrails, and PII detection.

In [1]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model = init_chat_model("groq:qwen/qwen3.8-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0'}}, client=<groq.resources.chat.completions.Completions object at 0x76b850b50650>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x76b8506e4b00>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

### Summarization middleware

 Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. summarization is useful for the following.


 - Long running conversation that exceed context windows.
 - Multi- turn dialogues with extensive history
 - Applications where preserving full conversation context matters

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
          model="groq:qwen/qwen3.8-27b",
          trigger=("messages", 10),
          keep=("messages", 4)
        )
    ]
)

In [3]:
### Run with thread id 

config={"configurable": {"thread_id" : "test-1"}}

In [4]:
# Alternative test data

# questions = [
#     "Why is the sky blue?",
#     "How do bees make honey?",
#     "Why do cats purr?",
#     "How do airplanes fly?",
#     "Why does ice float?",
#     "How do plants grow?",
# ]

questions = [
    "What is Java?",
    "What is HTML?",
    "What is 9+6?",
    "What is 18-9?",
    "What is 6*7?",
    "What is 81/9?",
]

for q in questions:
    response=agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is Java?', additional_kwargs={}, response_metadata={}, id='c9e2ec72-49a4-4de9-9391-a2627654b774'), AIMessage(content='**Java** is a widely used, high-level, general-purpose, **object-oriented programming language**. It was originally developed by James Gosling at Sun Microsystems (later acquired by Oracle) and released in 1995.\n\n### Key Characteristics of Java:\n\n1. **Object-Oriented**: Java is designed around the concept of objects and classes, promoting code reuse, modularity, and scalability.\n2. **Platform-Independent**: Java code is compiled into **bytecode**, which runs on the **Java Virtual Machine (JVM)**. This allows Java programs to run on any system with a compatible JVM, following the principle of *"Write Once, Run Anywhere" (WORA)*.\n3. **Robust and Secure**: Java includes features like automatic garbage collection, type checking, and exception handling to ensure reliability and security.\n4. **Multithreaded**: Java sup

### Token Size

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool

def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
          model="google_genai:gemini-3.5-flash-lite",
          trigger=("tokens", 250),
          keep=("tokens", 150)
        )
    ]
)

config = {"configurable" : {"thread_id" : "test-1"}}

#token counter {approximate}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return  total_chars

In [6]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~707 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='197e2116-b593-4e19-8dad-904a3993b11d'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_3020137287397603358': 'EpYBCpMBAWkUfRPTzQtKYB3xKYx0K0Oahm0jCqzzg0AMY4wfFlcZyur5rwI/+hb27r5NjUrHIh0ZEA62YjkLUMtPDvmezeDGiJab8LzYYHYR5N8EWVZ8RZIbORfGeoLDRzc9ophdg+WeURhSRkF3bdgDDv4Ixegpzyg52dP2cSicXmv7sf2spQ7pLY+ZJuktqWyCouJlitJq'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a21602-b244-70dc-8dad-3a942209389e-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_3020137287397603358', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens': 16, 'total_tokens': 69, 'input_token_de

In [7]:
from langchain.agents import create_agent 
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotel(city: str)-> str:
    """Search hotels."""
    return f"hotels in {city} : Grand Hotel $350. City Inn $180, Budget stay $75"

#Low fraction for testing

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=('fraction', 0.005),
            keep=('fraction',0.002)
        )
    ]
)

config = {"configurable" : {"thread_id": "test_1"}}

# Token Counter 

def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages)


#Test 
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens= count_tokens(response['messages'])
    fraction = tokens / 128000
    print(f"{city}: ~{tokens} tokens ({fraction: .4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~479 tokens ( 0.3742%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='95489021-8940-4ec3-a84d-f2e8996009d4'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotel', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_16468107231349052589': 'EpYBCpMBAWkUfRPWF8sQ9C7K7BSfCzGwfCdhI8OVC60hiJjZlZWyeBmKnQ0WCFsIn6qIYcTwXH/RjmrtCXQTdUPkTOjKqciSoN2kq58x26EJi4TohoaS7t9Ssa37dXB/Y9Ep/gx9jP3jCOxBnWREIKenQA/cW6mnzvkvcEt3RXXMGlFlprxw90WDIVkw79Zx6N9b6yUtuzJc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a21603-cfee-766f-819f-0cd34ddc1a2f-0', tool_calls=[{'name': 'search_hotel', 'args': {'city': 'Paris'}, 'id': 'call_16468107231349052589', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 16, 'total_tokens': 61, 'input_token_


### Human in the loop middleware

Pause agent execution for human approval, editing or rejection of tool calls before they execute. Huma in the loop is useful for the following 

- high - stakes operations requiring human approval
- compliance workflows where human oversight is mandatory
- long running conversations where human feedback guides the agent

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_mail_tool(email_id: str) -> str:
    """Mock functionm  to refad an email by its ID. """
    return f"Email content for ID: {email_id}"


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock Function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [9]:
agent=create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_mail_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [10]:
config = {"configurable": {"thread_id": "test-approve"}}

# step 1 - request

result = agent.invoke(
    {"messages": [HumanMessage(content='send email to john@test.com with subject "hello" and body "how are you?"')]},
    config=config
)

In [11]:
result

{'messages': [HumanMessage(content='send email to john@test.com with subject "hello" and body "how are you?"', additional_kwargs={}, response_metadata={}, id='2ee0d5c2-e24a-41ad-b11b-16af35e99e41'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "how are you?", "subject": "hello", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'call_6099992032380860742': 'EpYBCpMBAWkUfRMNHUdi4EckweXcFm+Yk2NM9hjhZahXn95Ss2pndkAhJkQYoHx3/dSxiNOZdCCgSxpGzdjqGqtZIxP/CZnmThh9kLi0nwvVq0Odi2y+hWBjXjhwyt60Ud1OiEYLLWltZi/2vIfwpXeNP4sChgEJoAyQeZrBXibCBLEG5Ro8LmrzGnmHp1lhW2Wd64/kjYU+'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a21604-334d-77e8-a89a-295e243dca83-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'how are you?', 'subject': 'hello', 'recipient': 'john@test.com'}, 'id': 'call_609999

In [12]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': "Email sent to john@test.com with subject 'hello'.", 'extras': {'signature': 'EpYBCpMBAWkUfRNkAWtNbda784/5rngEu3u5TJUdcXQjFRXf5E6v+xMehs3QqKaQhz8jPH4OyhAjM/72Fs5rbDxkujxXwABUXmGPhOK3GjLtEbVm8tZuCZXbZ+w6zvYo5ydDr/s0AtLbOCKvkp//ba2n4YnA9grAZOSUxKYJm51UXo4TorOB94QwGk+gmWMZEGXyl2QNbKfi'}}]


In [13]:
result

{'messages': [HumanMessage(content='send email to john@test.com with subject "hello" and body "how are you?"', additional_kwargs={}, response_metadata={}, id='2ee0d5c2-e24a-41ad-b11b-16af35e99e41'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "how are you?", "subject": "hello", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'call_6099992032380860742': 'EpYBCpMBAWkUfRMNHUdi4EckweXcFm+Yk2NM9hjhZahXn95Ss2pndkAhJkQYoHx3/dSxiNOZdCCgSxpGzdjqGqtZIxP/CZnmThh9kLi0nwvVq0Odi2y+hWBjXjhwyt60Ud1OiEYLLWltZi/2vIfwpXeNP4sChgEJoAyQeZrBXibCBLEG5Ro8LmrzGnmHp1lhW2Wd64/kjYU+'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a21604-334d-77e8-a89a-295e243dca83-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'how are you?', 'subject': 'hello', 'recipient': 'john@test.com'}, 'id': 'call_609999

### Reject

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_mail_tool(email_id:str) -> str:
    """MOck function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient:str, sunject: str, body: str)-> str:
    """Mock function to send an email."""
    return f"Emaiul sent to {recipient} with subject '{subject}'"


    agent = create_agent(
        model="google_genai:gemini-3.5-flash-lite",
        tools=[read_email_tool, send_email_tool],
        checkpointer=InMemorySaver(),
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "send_email_tool":{
                        "allowed_decisions": ["approve", "edit", "reject"],
                    },
                    "read_email_tool": False,
                }
            )
        ]
    ) 

In [15]:
config = {"configurable" : {"thread_id": "test-reject"}}
# step 1 : request

result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'how are you?'")]},
    config=config
)

In [16]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': 'The email was not sent because you rejected the tool call to send it. Let me know if you would like me to try again or help you with something else.', 'extras': {'signature': 'EpYBCpMBAWkUfRMj/J+euw1ro9M1wBJHjSd1+5ADlJdPYGDQ6V8kxITu1VuIS0zOJfPp/fvR++2NTwr9nzFepnOJjqv/D3S67fiNM07wfsCMTElPdA0sd1Rcdliw0N7DSn8nrpR+tq1WvkYswSTlPL+y4ZrvR9kKv4AE5QRJcai275bSSKWkSeO+QjrRAeDZU2PUQdlJHPvu'}}]


In [17]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='12e4c962-d396-4bee-a62e-1287a8782b4c'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "hello", "body": "how are you?", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'call_16107665438171768022': 'EpYBCpMBAWkUfRPXaWVfJC43Ochwt5oSUWr9beQ/WLOJqEQ+/L/gjN5YPS0gpsAUYGrAtknCftixMY73ZkeZj0vsfM4LpImCC5QD6qRrD49RISwniho4+0pgNowYi791vY8W+DCv4E6ZZbG2LvccEDlvpS76ufTZA/fxM+KIJrPVhwAN0lteBGXabYDBz8C8vaVbfoX4vkUk'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a21604-57ba-7746-9350-158a6fc02d93-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'hello', 'body': 'how are you?', 'recipient': 'john@test.com'}, 'id': 'call_16107

### Editing

In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its id."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject {subject}"


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_mail_tool": False,
            }
        )
    ]
)

In [19]:
config = {"configurable": {"thread_id": "test_edit"}}

# step 1: Request (with wrong info)

result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'test' and body 'hello'")]},
    config=config
)

result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'test' and body 'hello'", additional_kwargs={}, response_metadata={}, id='dc04422f-1417-402f-9f43-aebca04a657f'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "test", "body": "hello", "recipient": "wrong@email.com"}'}, '__gemini_function_call_thought_signatures__': {'call_11123869092503602696': 'EpYBCpMBAWkUfRN+TqtU7vZa7O82qhcJtc25VuLfoQ4cnXHvWdFiUORN9Qubr5B6sssMrVRagh3czt6JGAntcPukmUluzgzbWKk5pYEkkXdCvnl0kEIIKPsicktVRjVlyC2k34BAwnjYy8BaWBoPCEY2oZWO8NfElPhSzSjnWf3mewWAJbkN6W+MgZ6AbM027imak/3hqTQa'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a21621-281e-73c9-94e3-7a0a46a52793-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'test', 'body': 'hello', 'recipient': 'wrong@email.com'}, 'id': 'call_11123869092503602696', 

In [20]:
# step 2 : edit and approve

if"__interrupt__" in result:
    print("Paused Editing")

    result = agent.invoke(
        Command(
           resume={
             "decisions": [
                {
                    "type": "edit",
                    "edited_action":{
                        "name": "send_email_tool",
                        "args":{
                            "recipient": "correct@email.com",
                            "subject":"Corrected Subject",
                            "body": "This was edited by human before sending"
                        }
                    }
                }
             ]
        }),
        config=config
    )

    print(f"result: {result['messages'][-1].content}")

Paused Editing
result: [{'type': 'text', 'text': 'The email has been sent successfully. Note that a human reviewer intervened and updated the recipient to `correct@email.com`, the subject to `Corrected Subject`, and the body accordingly before sending.', 'extras': {'signature': 'EpYBCpMBAWkUfRNh4YdxzakSPIhi2efNURUNjlsisqwVbmGPmpjfMxa4A74Of74k8AofP9kH2PskEmX173POOg7nwWR2geNeWpz6aqxnSZue/GTVFYYbtIlYpCWwGBvBEym1wQ4GCqXUKy79EG2Gmt4GC5S4Ln4uNqhtgjkBQy1ZSWs1L0oLke2i7Uib285jovzZ9ioy1bKA'}}]


In [21]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'test' and body 'hello'", additional_kwargs={}, response_metadata={}, id='dc04422f-1417-402f-9f43-aebca04a657f'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "test", "body": "hello", "recipient": "wrong@email.com"}'}, '__gemini_function_call_thought_signatures__': {'call_11123869092503602696': 'EpYBCpMBAWkUfRN+TqtU7vZa7O82qhcJtc25VuLfoQ4cnXHvWdFiUORN9Qubr5B6sssMrVRagh3czt6JGAntcPukmUluzgzbWKk5pYEkkXdCvnl0kEIIKPsicktVRjVlyC2k34BAwnjYy8BaWBoPCEY2oZWO8NfElPhSzSjnWf3mewWAJbkN6W+MgZ6AbM027imak/3hqTQa'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a21621-281e-73c9-94e3-7a0a46a52793-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'test', 'body': 'hello', 'recipient': 'wrong@email.com'}, 'id': 'call_11123869092503602696', 